### LIBRARY IMPORTS

In [9]:
import yaml
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, TensorDataset

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_loader import DataLoader
from src.nn_regressor import NNRegressor

### CONFIGURATION

In [10]:
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

active_dataset = "scores"
active_dataset_config = config["datasets"][active_dataset]

data_loader = DataLoader(active_dataset, active_dataset_config["id_column"], active_dataset_config["target"])
data, _ = data_loader.load_processed_data()

X_train, X_test, y_train, y_test = train_test_split(
    data.drop(columns=[active_dataset_config["target"]]),
    data[active_dataset_config["target"]],
    test_size = 0.8,
    random_state = 42
)

scaler = RobustScaler()
y_train = pd.Series(scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten(), index=y_train.index)
y_test = pd.Series(scaler.transform(y_test.values.reshape(-1, 1)).flatten(), index=y_test.index)

### RIDGE & LASSO CROSS-VALIDATION

In [11]:
ridge = RidgeCV(alphas=[0.001, 0.1, 1, 10, 100, 500, 1_000, 5_000, 10_000], cv=5)
ridge.fit(X_train, y_train)

ridge_test_preds = ridge.predict(X_test)

print(f"Ridge best hyperparameter: {ridge.alpha_}")
print(f"Ridge validation MSE: {mean_squared_error(ridge_test_preds, y_test)}")

print('-' * 50)

lasso = LassoCV(alphas=[100, 500, 1_000, 5_000, 10_000], cv=5)
lasso.fit(X_train, y_train)

lasso_test_preds = lasso.predict(X_test)

print(f"Lasso best hyperparameter: {ridge.alpha_}")
print(f"Lasso validation MSE: {mean_squared_error(lasso_test_preds, y_test)}")

Ridge best hyperparameter: 1.0
Ridge validation MSE: 0.10651546418151875
--------------------------------------------------
Lasso best hyperparameter: 1.0
Lasso validation MSE: 0.4806114492803612


### RANDOM FOREST

In [12]:
rf = RandomForestRegressor(n_estimators=1_000, max_depth=5, max_features="sqrt")
rf.fit(X_train, y_train)

rf_test_preds = rf.predict(X_test)

print(f"Random forest validation MSE: {mean_squared_error(rf_test_preds, y_test)}")

Random forest validation MSE: 0.20517646929127858


### NEURAL NETWORK

In [22]:
class MLPRegressor(NNRegressor):
    def __init__(self, hidden_size=64, batch_size=64, dropout=0.3):
        super().__init__(hidden_size, batch_size)
        self.dropout = dropout

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        epochs: int = 100,
        patience: int = 20,
        lr: float = 1e-3
    ) -> None:
        input_size = X_train.shape[1]
        output_size = y_train.shape[1] if len(y_train.shape) > 1 else 1
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.float32).view(-1, output_size).to(self.device)

        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.view_as(preds))
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()
                scheduler.step(val_loss)

            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch + 1}. Validation MSE: {val_loss:.6f}. Learning rate: {current_lr}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                print(f"Early stopping on epoch {epoch + 1}!! Best validation MSE: {best_loss:.6f}")
                break
                
        if best_model:
            self.load_state_dict(best_model)

    def _get_network(self, input_size: int, output_size: int) -> None:
        self.network = nn.Sequential(
            nn.Linear(input_size, self.hidden_size),
            nn.BatchNorm1d(self.hidden_size),
            nn.GELU(),
            nn.Dropout(self.dropout),

            nn.Linear(self.hidden_size, self.hidden_size),
            nn.BatchNorm1d(self.hidden_size),
            nn.GELU(),
            nn.Dropout(self.dropout),

            nn.Linear(self.hidden_size, output_size)
        )

        self.network.to(self.device)

mlp_regressor = MLPRegressor()
mlp_regressor.fit(X_train.values, y_train.values, X_test.values, y_test.values)

Epoch 1. Validation MSE: 0.115339. Learning rate: 0.001
Epoch 2. Validation MSE: 0.115031. Learning rate: 0.001
Epoch 3. Validation MSE: 0.110940. Learning rate: 0.001
Epoch 4. Validation MSE: 0.111024. Learning rate: 0.001
Epoch 5. Validation MSE: 0.109819. Learning rate: 0.001
Epoch 6. Validation MSE: 0.111850. Learning rate: 0.001
Epoch 7. Validation MSE: 0.112659. Learning rate: 0.001
Epoch 8. Validation MSE: 0.107926. Learning rate: 0.001
Epoch 9. Validation MSE: 0.108465. Learning rate: 0.001
Epoch 10. Validation MSE: 0.107955. Learning rate: 0.001
Epoch 11. Validation MSE: 0.107653. Learning rate: 0.001
Epoch 12. Validation MSE: 0.108298. Learning rate: 0.001
Epoch 13. Validation MSE: 0.108856. Learning rate: 0.001
Epoch 14. Validation MSE: 0.109120. Learning rate: 0.001
Epoch 15. Validation MSE: 0.109265. Learning rate: 0.001
Epoch 16. Validation MSE: 0.108298. Learning rate: 0.001
Epoch 17. Validation MSE: 0.107645. Learning rate: 0.0005
Epoch 18. Validation MSE: 0.108994. Lea